# Configuration Gemini API

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma

/var/folders/ts/h007gk3j78311whxhv103ypr0000gn/T/ipykernel_99957/846644173.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


### Configure API key

In [16]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

### Basic steps
LLMs are trained offline on a large corpus of public data. Hence they cannot answer questions based on custom or private data accurately without additional context.

If you want to make use of LLMs to answer questions based on private data, you have to provide the relevant documents as context alongside your prompt. This approach is called Retrieval Augmented Generation (RAG).

You will use this approach to create a question-answering assistant using the Gemini text model integrated through LangChain. The assistant is expected to answer questions about the Gemini model. To make this possible you will add more context to the assistant using data from a website.

In this tutorial, you'll implement the two main components in an RAG-based architecture:

1. Retriever

    Based on the user's query, the retriever retrieves relevant snippets that add context from the document. In this tutorial, the document is the website data.
    The relevant snippets are passed as context to the next stage - "Generator".

2. Generator

    The relevant snippets from the website data are passed to the LLM along with the user's query to generate accurate answers.

You'll learn more about these stages in the upcoming sections while implementing the application.

## Retriever

In this stage, you will perform the following steps:

1. Read and parse the website data using LangChain.

2. Create embeddings of the website data.

    Embeddings are numerical representations (vectors) of text. Hence, text with similar meaning will have similar embedding vectors. You'll make use of Gemini's embedding model to create the embedding vectors of the website data.

3. Store the embeddings in Chroma's vector store.
    
    Chroma is a vector database. The Chroma vector store helps in the efficient retrieval of similar vectors. Thus, for adding context to the prompt for the LLM, relevant embeddings of the text matching the user's question can be retrieved easily using Chroma.

4. Create a Retriever from the Chroma vector store.

    The retriever will be used to pass relevant website embeddings to the LLM along with user queries.

### Read and parse the website data

LangChain provides a wide variety of document loaders. To read the website data as a document, you will use the `WebBaseLoader` from LangChain.

To know more about how to read and parse input data from different sources using the document loaders of LangChain, read LangChain's [document loaders guide](https://python.langchain.com/docs/integrations/document_loaders).

In [5]:
loader = WebBaseLoader("https://minecraft.fandom.com/fr/wiki/Minecraft_Wiki")
docs = loader.load()

If you only want to select a specific portion of the website data to add context to the prompt, you can use regex, text slicing, or text splitting.

In this example, you'll use Python's `split()` function to extract the required portion of the text. The extracted text should be converted back to LangChain's `Document` format.

In [6]:
# Extract the text from the website data document
text_content = docs[0].page_content

# For Wikipedia, we need to clean up the navigation noise and get the actual article content
# Remove excessive whitespace and empty lines
lines = [line.strip() for line in text_content.split('\n') if line.strip()]
cleaned_text = '\n'.join(lines)

# Find where the actual article content starts (usually after menu items)
# Take content from the first substantial paragraph onwards
# Look for the first line that seems to be actual content (not navigation)
start_idx = 0
for i, line in enumerate(lines):
    if len(line) > 50 and not any(nav_word in line.lower() for nav_word in ['menu', 'navigation', 'rechercher', 'connexion', 'déplacer']):
        start_idx = i
        break

# Take a good chunk of the article content (first 20000 chars after cleaning is usually ~4000-5000 words)
article_content = '\n'.join(lines[start_idx:start_idx+150])

# Limit to reasonable size for embeddings
final_text = article_content[:15000]

# Convert the text to LangChain's `Document` format
docs = [Document(page_content=final_text, metadata={"source": "Wikipedia Minecraft"})]

### Initialize Gemini's embedding model

To create the embeddings from the website data, you'll use Gemini's embedding model, **gemini-embedding-001** which supports creating text embeddings.

To use this embedding model, you have to import `GoogleGenerativeAIEmbeddings` from LangChain. To know more about the embedding model, read Google AI's [language documentation](https://ai.google.dev/models/gemini).

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

### Store the data using Chroma

To create a Chroma vector database from the website data, you will use the `from_documents` function of `Chroma`. Under the hood, this function creates embeddings from the documents created by the document loader of LangChain using any specified embedding model and stores them in a Chroma vector database.  

You have to specify the `docs` you created from the website data using LangChain's `WebBasedLoader` and the `gemini_embeddings` as the embedding model when invoking the `from_documents` function to create the vector database from the website data. You can also specify a directory in the `persist_directory` argument to store the vector store on the disk. If you don't specify a directory, the data will be ephemeral in-memory.


In [8]:
# Save to disk
vectorstore = Chroma.from_documents(
                     documents=docs,                 # Data
                     embedding=gemini_embeddings,    # Embedding model
                     persist_directory="./chroma_db" # Directory to save data
                     )

### Create a retriever using Chroma

You'll now create a retriever that can retrieve website data embeddings from the newly created Chroma vector store. This retriever can be later used to pass embeddings that provide more context to the LLM for answering user's queries.


To load the vector store that you previously stored in the disk, you can specify the name of the directory that contains the vector store in `persist_directory` and the embedding model in the `embedding_function` arguments of Chroma's initializer.

You can then invoke the `as_retriever` function of `Chroma` on the vector store to create a retriever.

In [9]:
# Load from disk
vectorstore_disk = Chroma(
                        persist_directory="./chroma_db",       # Directory of db
                        embedding_function=gemini_embeddings   # Embedding model
                   )
# Get the Retriever interface for the store to use later.
# When an unstructured query is given to a retriever it will return documents.
# Read more about retrievers in the following link.
# https://python.langchain.com/docs/modules/data_connection/retrievers/
#
# Since only 1 document is stored in the Chroma vector store, search_kwargs `k`
# is set to 1 to decrease the `k` value of chroma's similarity search from 4 to
# 1. If you don't pass this value, you will get a warning.
retriever = vectorstore_disk.as_retriever(search_kwargs={"k": 1})

# Check if the retriever is working by trying to fetch the relevant docs related
# to the word 'MMLU' (Massive Multitask Language Understanding). If the length is greater than zero, it means that
# the retriever is functioning well.
print(len(retriever.invoke("MMLU")))

/var/folders/ts/h007gk3j78311whxhv103ypr0000gn/T/ipykernel_99957/2569891724.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_disk = Chroma(


1


## Test rapide

Voici un test simple pour vérifier que le retriever fonctionne avec les données Minecraft :

In [10]:
# Test simple : récupérer les documents pertinents pour une question Minecraft
test_question = "Qu'est-ce que Minecraft?"
relevant_docs = retriever.invoke(test_question)
print(f"Question : {test_question}")
print(f"Nombre de documents trouvés : {len(relevant_docs)}")
if relevant_docs:
    print("\n--- Contenu pertinent trouvé ---")
    print(relevant_docs[0].page_content[:500])

Question : Qu'est-ce que Minecraft?
Nombre de documents trouvés : 1

--- Contenu pertinent trouvé ---
Le Minecraft Wiki – La principale source d'informations sur Minecraft
Se connecter
Créer un compte gratuit
Minecraft Wiki
Explorer
Page d'accueil
Toutes les pages
Cartes interactives
Minecraft Wiki
Projets
Communauté
Bureaux de l'administration
Les règles
Aide
Discord du wiki
Twitter du wiki
Minecraft
Site web
Support, aide
Suivi des bugs
Commentaires
Traduction du jeu
Réseaux sociaux
Twitter
Facebook
YouTube
Twitch
Instagram
Discord
Minecraft-France.fr
Pages utiles
Blocs
Objets
Créatures
Biomes


In [11]:
# Test avec le modèle Gemini complet : obtenir une réponse
# D'abord, initialize le modèle et le prompt
from langchain_google_genai import ChatGoogleGenerativeAI

llm_test = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

llm_prompt_template = """Tu es un assistant pour répondre aux questions.
Utilise le contexte suivant pour répondre.
Si tu ne connais pas la réponse, dis simplement que tu ne sais pas.
Réponds en maximum 3 phrases.\n
Question: {question} 
Contexte: {context} 
Réponse:"""

llm_prompt_test = PromptTemplate.from_template(llm_prompt_template)

# Créer la chaîne RAG de test
from langchain_core.output_parsers import StrOutputParser

rag_chain_test = (
    {"context": retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)), 
     "question": RunnablePassthrough()}
    | llm_prompt_test
    | llm_test
    | StrOutputParser()
)

# Tester avec une question simple
test_question_minecraft = "C'est quoi Minecraft?"
print(f"Question: {test_question_minecraft}")
print("\nRéponse:")
try:
    response = rag_chain_test.invoke(test_question_minecraft)
    print(response)
except Exception as e:
    print(f"Erreur: {e}")

Question: C'est quoi Minecraft?

Réponse:
Erreur: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 33.612721363s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gem

## Generator

The Generator prompts the LLM for an answer when the user asks a question. The retriever you created in the previous stage from the Chroma vector store will be used to pass relevant embeddings from the website data to the LLM to provide more context to the user's query.

You'll perform the following steps in this stage:

1. Chain together the following:
    * A prompt for extracting the relevant embeddings using the retriever.
    * A prompt for answering any question using LangChain.
    * An LLM model from Gemini for prompting.
    
2. Run the created chain with a question as input to prompt the model for an answer.


### Initialize Gemini

You must import `ChatGoogleGenerativeAI` from LangChain to initialize your model.
 In this example, you will use **gemini-2.0-flash**, as it supports text summarization. To know more about the text model, read Google AI's [language documentation](https://ai.google.dev/models/gemini).

You can configure the model parameters such as ***temperature*** or ***top_p***,  by passing the appropriate values when initializing the `ChatGoogleGenerativeAI` LLM.  To learn more about the parameters and their uses, read Google AI's [concepts guide](https://ai.google.dev/docs/concepts#model_parameters).

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

# To configure model parameters use the `generation_config` parameter.
# eg. generation_config = {"temperature": 0.7, "topP": 0.8, "topK": 40}
# If you only want to set a custom temperature for the model use the
# "temperature" parameter directly.

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

### Create prompt templates

You'll use LangChain's [PromptTemplate](https://python.langchain.com/docs/how_to/#prompt-templates) to generate prompts to the LLM for answering questions.

In the `llm_prompt`, the variable `question` will be replaced later by the input question, and the variable `context` will be replaced by the relevant text from the website retrieved from the Chroma vector store.

In [13]:
# Prompt template to query Gemini
llm_prompt_template = """You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you don't know the answer, just say that you don't know.
Use five sentences maximum and keep the answer concise.\n
Question: {question} \nContext: {context} \nAnswer:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template="You are an assistant for question-answering tasks.\nUse the following context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse five sentences maximum and keep the answer concise.\n\nQuestion: {question} \nContext: {context} \nAnswer:"


### Create a stuff documents chain

LangChain provides [Chains](https://python.langchain.com/docs/modules/chains/) for chaining together LLMs with each other or other components for complex applications. You will create a **stuff documents chain** for this application. A stuff documents chain lets you combine all the relevant documents, insert them into the prompt, and pass that prompt to the LLM.

You can create a stuff documents chain using the [LangChain Expression Language (LCEL)](https://python.langchain.com/docs/expression_language).

To learn more about different types of document chains, read LangChain's [chains guide](https://python.langchain.com/docs/modules/chains/document/).

The stuff documents chain for this application retrieves the relevant website data and passes it as the context to an LLM prompt along with the input question.

In [14]:
# Combine data from documents to readable string format.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create stuff documents chain using LCEL.
#
# This is called a chain because you are chaining together different elements
# with the LLM. In the following example, to create the stuff chain, you will
# combine the relevant context from the website data matching the question, the
# LLM model, and the output parser together like a chain using LCEL.
#
# The chain implements the following pipeline:
# 1. Extract the website data relevant to the question from the Chroma
#    vector store and save it to the variable `context`.
# 2. `RunnablePassthrough` option to provide `question` when invoking
#    the chain.
# 3. The `context` and `question` are then passed to the prompt where they
#    are populated in the respective variables.
# 4. This prompt is then passed to the LLM (`gemini-2.0-flash`).
# 5. Output from the LLM is passed through an output parser
#    to structure the model's response.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### Prompt the model

You can now query the LLM by passing any question to the `invoke()` function of the stuff documents chain you created previously.

In [15]:
from IPython.display import Markdown

Markdown(rag_chain.invoke("What is Gemini?"))

I do not know what Gemini is based on the provided context. The context only contains navigation links and information related to the Minecraft Wiki.